# Data exploration

First contact with an image-mask pair : `image_1.png` and `mask_1.png`. This answers the question: what does our dataset look like ?

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from oc_p02_fashion_segmentation.constants import (
    ID_TO_CLASS,
    NUM_CLASSES,
)
from oc_p02_fashion_segmentation.visualization import (
    CLASS_CMAP,
    class_legend_handles,
    display_segmentation,
)

DATA_DIR = Path("../data/input")
IMAGE_PATH = DATA_DIR / "IMG" / "image_1.png"
MASK_PATH = DATA_DIR / "Mask" / "mask_1.png"

## Load with pillow and convert to numpy array

In [ ]:
image = Image.open(IMAGE_PATH)
mask = Image.open(MASK_PATH)

image_array = np.array(image)
mask_array = np.array(mask)

print(
    f"Image : mode={image.mode}, shape={image_array.shape}, dtype={image_array.dtype}"
)
print(f"Mask : mode={mask.mode}, shape={mask_array.shape}, dtype={mask_array.dtype}")

## Mask distinct values

Each pixel of the mask is an integer representing a class. Let's check the distinct values and their counts.

In [ ]:
values, counts = np.unique(mask_array, return_counts=True)
print(f"Mask distinct values : {values}")
for value, count in zip(values, counts):
    class_name = ID_TO_CLASS.get(value, "Unknown")
    print(f"  {class_name} : {count:>6} pixels ({count / mask_array.size:.1%})")

## Side-by-side display of image and mask

The mask is displayed with a categorical colormap, `CLASS_CMAP`: without `cmap`, matplotlib would use `viridis`, where neighboring classes (e.g., 9 and 10) have almost identical colors.

By default, `imshow` stretches the colormap between the min and max values present in the array, so the same class could get a different color from one mask to another. `vmin` and `vmax` pin the scale to the full range of class ids (0 to 17), which keeps the color code consistent across all images.

In [ ]:
display_segmentation(image_array, mask_array)

## Grid display of several image-mask pairs

Each pair takes two adjacent columns (image, then mask). The same `CLASS_CMAP` with fixed `vmin` / `vmax` is used for every mask, so a class has the same color everywhere in the grid. A single shared legend lists the classes present in the displayed masks.

In [ ]:
def show_pairs(indices, pairs_per_row=3):
    n_rows = math.ceil(len(indices) / pairs_per_row)
    fig, axes = plt.subplots(
        n_rows,
        2 * pairs_per_row,
        figsize=(3 * pairs_per_row * 2, 4.5 * n_rows),
        squeeze=False,
    )

    present_classes = set()
    for ax in axes.flat:
        ax.axis("off")

    for position, index in enumerate(indices):
        row, col = divmod(position, pairs_per_row)
        image_array = np.array(Image.open(DATA_DIR / "IMG" / f"image_{index}.png"))
        mask_array = np.array(Image.open(DATA_DIR / "Mask" / f"mask_{index}.png"))
        present_classes.update(np.unique(mask_array).tolist())

        ax_image, ax_mask = axes[row, 2 * col], axes[row, 2 * col + 1]
        ax_image.imshow(image_array)
        ax_image.set_title(f"image_{index}")
        ax_mask.imshow(
            mask_array,
            cmap=CLASS_CMAP,
            vmin=0,
            vmax=NUM_CLASSES - 1,
            interpolation="nearest",
        )
        ax_mask.set_title(f"mask_{index}")

    legend_handles = class_legend_handles(present_classes)
    plt.tight_layout()
    fig.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 0),
        ncol=min(len(legend_handles), 9),
        frameon=False,
    )
    plt.show()

In [ ]:
# Show the first 10 image-mask pairs
show_pairs(range(50), pairs_per_row=2)

## Notes on the dataset

### Annotation errors

* Pair 1: `Upper-clothes` strips extend from the coat and run across the wall to the right edge.
* Pair 26: the dress has small holes in the mask, which are not present in the image.
* Pair 29: the top and the shorts under the vest are annotated as a single `Dress`.

### Annotation conventions

* Tights -> `Pants`: pairs 1, 2, 8, 31, 38, among others.
* Shorts -> `Pants`: pairs 6, 14, 22, 23, 47.
* Hat held in hand -> `Hat` (pair 45); scarf on the head -> `Scarf` (pair 26).
* Prescription glasses -> no class: they remain in `Face` (pair 17). Only sunglasses have their own class.

### Unusual input images

* Pair 9: the image is only 97 x 412 pixels, with a watermark at the top. All the others are about 400 x 600.
* Pairs 10 and 11: rounded corners and a "TOMI" watermark.
* Pair 13: the image is in black and white.
* Pair 21: it's a close-up; the person is cropped off at mid-thigh.

## Pairs with faulty masks

Pairs 1, 26 and 29 have annotation errors (see the notes above). They are kept in the dataset as a sanity check: the segmentation scores should be low on them. `FAULTY_MASK_INDICES` lets us compute the metrics both with and without these pairs.

In [ ]:
ALL_INDICES = sorted(
    int(path.stem.removeprefix("image_"))
    for path in (DATA_DIR / "IMG").glob("image_*.png")
)
FAULTY_MASK_INDICES = frozenset({1, 26, 29})
CLEAN_INDICES = [index for index in ALL_INDICES if index not in FAULTY_MASK_INDICES]

print(
    f"All pairs : {len(ALL_INDICES)}, faulty : {sorted(FAULTY_MASK_INDICES)}, clean : {len(CLEAN_INDICES)}"
)
show_pairs(sorted(FAULTY_MASK_INDICES))